# Decorator Pattern
## Design Patterns for the AI Era · Part 5

> **Bad Code → Better Design → Working Python → AI Application**

Add cross-cutting behaviour without modifying the core function. Wrap every LLM call with logging, retry, caching, and cost tracking.

## 1. The Problem

Every LLM call needs logging, timing, retry, and cost tracking. The naive approach: copy-paste the boilerplate into every function.

In [ ]:
# ❌ BAD CODE — boilerplate duplicated everywhere
import time

def ask_openai_bad(prompt: str) -> str:
    t0 = time.time()
    print(f'[LOG] Calling OpenAI: {prompt[:40]}')
    # Fake call
    time.sleep(0.05)
    result = f'[OpenAI] {prompt[:30]}'
    elapsed = time.time() - t0
    print(f'[LOG] Done in {elapsed:.3f}s')
    return result

def ask_anthropic_bad(prompt: str) -> str:
    t0 = time.time()                                   # copy
    print(f'[LOG] Calling Anthropic: {prompt[:40]}')  # copy
    time.sleep(0.03)
    result = f'[Anthropic] {prompt[:30]}'
    elapsed = time.time() - t0                         # copy
    print(f'[LOG] Done in {elapsed:.3f}s')             # copy
    return result
# Adding Ollama = copy again 😱

ask_openai_bad('What is a design pattern?')

## 2. What's Wrong?

- Boilerplate duplicated across every LLM function
- Changing retry logic requires editing every function
- Cannot compose concerns (add caching without touching all functions)
- Cannot test cross-cutting concerns in isolation

In [ ]:
# ✅ GOOD CODE — Decorator Pattern
import time
import functools
from typing import Callable

LLMFn = Callable[[str], str]


def with_logging(fn: LLMFn) -> LLMFn:
    @functools.wraps(fn)
    def wrapper(prompt: str) -> str:
        print(f'→ {fn.__name__}({prompt[:50]!r})')
        result = fn(prompt)
        print(f'← {fn.__name__}: {result[:50]!r}')
        return result
    return wrapper


def with_timing(fn: LLMFn) -> LLMFn:
    @functools.wraps(fn)
    def wrapper(prompt: str) -> str:
        t0 = time.perf_counter()
        result = fn(prompt)
        print(f'⏱  {fn.__name__}: {time.perf_counter() - t0:.4f}s')
        return result
    return wrapper


def with_retry(attempts: int = 3, delay: float = 0.1):
    def decorator(fn: LLMFn) -> LLMFn:
        @functools.wraps(fn)
        def wrapper(prompt: str) -> str:
            for i in range(attempts):
                try:
                    return fn(prompt)
                except Exception as exc:
                    if i == attempts - 1:
                        raise
                    print(f'⚠ Retry {i+1}/{attempts}: {exc}')
                    time.sleep(delay * (2 ** i))
        return wrapper
    return decorator


def with_cache(fn: LLMFn) -> LLMFn:
    _cache: dict[str, str] = {}

    @functools.wraps(fn)
    def wrapper(prompt: str) -> str:
        if prompt not in _cache:
            print(f'💾 cache miss — calling {fn.__name__}')
            _cache[prompt] = fn(prompt)
        else:
            print(f'💾 cache hit')
        return _cache[prompt]
    return wrapper


# The raw function — no concerns mixed in
def raw_llm(prompt: str) -> str:
    time.sleep(0.02)  # simulate latency
    return f'[LLM] {prompt[:60]}'


# Compose decorators — outermost runs first
ask = with_logging(with_timing(with_retry(attempts=2)(with_cache(raw_llm))))

print('=== First call ===')
r1 = ask('What is the Decorator pattern?')
print()
print('=== Second call (same prompt) ===')
r2 = ask('What is the Decorator pattern?')

## 4. Tests

In [ ]:
# ✅ Tests — each decorator tested in isolation

def test_logging_calls_wrapped_fn():
    calls = []
    def fake_llm(prompt): calls.append(prompt); return 'ok'
    logged = with_logging(fake_llm)
    result = logged('hello')
    assert result == 'ok'
    assert calls == ['hello']
    print('PASS test_logging_calls_wrapped_fn')

def test_cache_prevents_second_call():
    call_count = [0]
    def counting_llm(prompt):
        call_count[0] += 1
        return 'result'
    cached = with_cache(counting_llm)
    cached('prompt')
    cached('prompt')
    assert call_count[0] == 1  # only called once
    print('PASS test_cache_prevents_second_call')

def test_retry_on_failure():
    attempts = [0]
    def flaky_llm(prompt):
        attempts[0] += 1
        if attempts[0] < 3:
            raise ValueError('transient error')
        return 'ok'
    retried = with_retry(attempts=3, delay=0.001)(flaky_llm)
    result = retried('hello')
    assert result == 'ok'
    assert attempts[0] == 3
    print('PASS test_retry_on_failure')

def test_retry_raises_after_exhaustion():
    def always_fail(prompt): raise RuntimeError('boom')
    retried = with_retry(attempts=2, delay=0.001)(always_fail)
    try:
        retried('x')
        assert False
    except RuntimeError:
        pass
    print('PASS test_retry_raises_after_exhaustion')

test_logging_calls_wrapped_fn()
test_cache_prevents_second_call()
test_retry_on_failure()
test_retry_raises_after_exhaustion()
print('\nAll tests passed ✅')

## 5. AI Example — PII Guardrail + Cost Tracker

In [ ]:
# 🤖 AI EXAMPLE — PII guardrail and token cost tracking
import re

PII_PATTERN = re.compile(r'\b\d{3}-\d{2}-\d{4}\b')  # SSN pattern

total_tokens = [0]

def with_pii_guard(fn: LLMFn) -> LLMFn:
    @functools.wraps(fn)
    def wrapper(prompt: str) -> str:
        if PII_PATTERN.search(prompt):
            raise ValueError('PII detected in prompt — blocked by guardrail')
        result = fn(prompt)
        if PII_PATTERN.search(result):
            return '[REDACTED — PII in response]'
        return result
    return wrapper


def with_cost_tracking(fn: LLMFn) -> LLMFn:
    @functools.wraps(fn)
    def wrapper(prompt: str) -> str:
        result = fn(prompt)
        # Rough token estimate
        tokens = len(prompt.split()) + len(result.split())
        total_tokens[0] += tokens
        print(f'💰 tokens this call: {tokens} | total: {total_tokens[0]}')
        return result
    return wrapper


safe_ask = with_pii_guard(with_cost_tracking(with_cache(raw_llm)))

print('=== Safe call ===')
safe_ask('What is the Strategy pattern?')

print('\n=== PII blocked ===')
try:
    safe_ask('My SSN is 123-45-6789, help me')
except ValueError as e:
    print(f'Blocked: {e}')

## 6. Summary

| | Before | After |
|---|---|---|
| Adding logging | Edit every LLM function | `with_logging(fn)` |
| Changing retry logic | Edit every function | Edit `with_retry` once |
| Composing concerns | Copy-paste | Chain decorators |
| Testing concerns | Entangled | Each decorator standalone |

**Decorator is the most immediately useful pattern for AI developers.**